# Batch processor

> Pad every SMILES sequence to ``max_len`` and build the two buffers the graph expects.


In [ ]:
#| default_exp batch_processor


In [ ]:
#| hide
from nbdev.showdoc import *


## Why batching is boring (and important)

The graph is compiled for shape `[batch, 202]`. Real molecules are shorter. So before `execute`, we:

1. collect token ids from each context,
2. right-pad with the pad token to `max_len`,
3. build an attention mask (`1` = real token, `0` = pad),
4. wrap both as device `Buffer`s inside `SmiTedInputs`.

That's it. No KV cache. No chunking. Fixed-length encoder batching.


In [ ]:
#| export
from __future__ import annotations

from typing import TYPE_CHECKING

import numpy as np

from max.driver import Buffer
from max.pipelines.lib.interfaces.batch_processor import (
    PaddedEncoderBatchProcessor,
    single_replica_context_batch,
)

if TYPE_CHECKING:
    from mat_gram01.model import SmiTedInputs


In [ ]:
#| export
class SmiTedBatchProcessor(PaddedEncoderBatchProcessor["SmiTedInputs"]):
    "Pads every batch item to SMI-TED's fixed ``max_len`` (202)."

    def prepare_initial_token_inputs(
        self,
        replica_batches,
        kv_cache_inputs=None,
        return_n_logits: int = 1,
    ):
        del kv_cache_inputs, return_n_logits
        context_batch = single_replica_context_batch(
            replica_batches,
            processor_name=type(self).__qualname__,
        )
        device0 = self.runtime.devices[0]
        tokens = [ctx.tokens.active for ctx in context_batch]
        pad_value = self._pad_token_id()
        max_len = self.config.get_max_seq_len()

        padded_tokens: list[np.ndarray] = []
        for token_ids in tokens:
            arr = np.asarray(token_ids, dtype=np.int64)
            if arr.ndim != 1:
                raise ValueError(f"expected 1-D token ids, got shape {arr.shape}")
            if len(arr) > max_len:
                raise ValueError(
                    f"sequence length {len(arr)} exceeds SMI-TED max_len {max_len}"
                )
            if len(arr) < max_len:
                arr = np.pad(
                    arr,
                    (0, max_len - len(arr)),
                    mode="constant",
                    constant_values=pad_value,
                )
            padded_tokens.append(arr)

        next_tokens_batch = np.stack(padded_tokens, axis=0)
        attention_mask = (next_tokens_batch != pad_value).astype(np.float32)
        return self._make_inputs(
            next_tokens_batch=Buffer.from_numpy(next_tokens_batch).to(device0),
            attention_mask=Buffer.from_numpy(attention_mask).to(device0),
        )

    def _make_inputs(
        self,
        *,
        next_tokens_batch: Buffer,
        attention_mask: Buffer,
    ) -> SmiTedInputs:
        from mat_gram01.model import SmiTedInputs

        return SmiTedInputs(
            next_tokens_batch=next_tokens_batch,
            attention_mask=attention_mask,
        )


### Checkpoint

*Batching's only job: turn ragged token lists into `[B, 202]` tokens + mask.*


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
